In [1]:
import os
import cv2
import pandas as pd
from pathlib import Path

In [2]:
DATA_DIR = Path("../data")

domains = ["artificial_jewellery", "home_cleaning", "shop"]

for domain in domains:
    print(domain, "exists:", (DATA_DIR / domain).exists())

artificial_jewellery exists: True
home_cleaning exists: True
shop exists: True


In [3]:
video_extensions = [".mp4"]

video_records = []

for domain in domains:
    domain_path = DATA_DIR / domain
    
    for file in domain_path.iterdir():
        if file.suffix.lower() in video_extensions:
            video_records.append({
                "domain": domain,
                "video_name": file.name,
                "video_path": str(file)
            })

df_videos = pd.DataFrame(video_records)
df_videos

,domain,video_name,video_path
0,artificial_jewellery,video_20260404_114752.mp4,../data/artificial_jewellery/video_20260404_11...
1,artificial_jewellery,video_20260404_120254.mp4,../data/artificial_jewellery/video_20260404_12...
2,artificial_jewellery,video_20260404_121005.mp4,../data/artificial_jewellery/video_20260404_12...
3,artificial_jewellery,video_20260404_115542.mp4,../data/artificial_jewellery/video_20260404_11...
4,artificial_jewellery,video_20260404_122845.mp4,../data/artificial_jewellery/video_20260404_12...
5,artificial_jewellery,video_20260405_105612_edit.mp4,../data/artificial_jewellery/video_20260405_10...
6,artificial_jewellery,video_20260404_121903.mp4,../data/artificial_jewellery/video_20260404_12...
7,home_cleaning,video_20260404_073018.mp4,../data/home_cleaning/video_20260404_073018.mp4
8,home_cleaning,video_20260404_074030.mp4,../data/home_cleaning/video_20260404_074030.mp4
9,shop,video_20260405_163219_edit.mp4,../data/shop/video_20260405_163219_edit.mp4


In [4]:
def get_video_metadata(video_path):
    cap = cv2.VideoCapture(video_path)
    
    if not cap.isOpened():
        return None
    
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    duration = frame_count / fps if fps > 0 else 0
    
    cap.release()
    
    return {
        "fps": fps,
        "frame_count": frame_count,
        "width": width,
        "height": height,
        "duration_sec": duration
    }

In [5]:
metadata_list = []

for path in df_videos["video_path"]:
    meta = get_video_metadata(path)
    metadata_list.append(meta)

meta_df = pd.DataFrame(metadata_list)
df_videos = pd.concat([df_videos, meta_df], axis=1)
df_videos

,domain,video_name,video_path,fps,frame_count,width,height,duration_sec
0,artificial_jewellery,video_20260404_114752.mp4,../data/artificial_jewellery/video_20260404_11...,30.008118,12127,1920,1080,404.123978
1,artificial_jewellery,video_20260404_120254.mp4,../data/artificial_jewellery/video_20260404_12...,30.005083,9359,1920,1080,311.913822
2,artificial_jewellery,video_20260404_121005.mp4,../data/artificial_jewellery/video_20260404_12...,30.002720,3864,1920,1080,128.788322
3,artificial_jewellery,video_20260404_115542.mp4,../data/artificial_jewellery/video_20260404_11...,30.008352,12150,1920,1080,404.887278
4,artificial_jewellery,video_20260404_122845.mp4,../data/artificial_jewellery/video_20260404_12...,29.895380,10666,1920,1080,356.777533
5,artificial_jewellery,video_20260405_105612_edit.mp4,../data/artificial_jewellery/video_20260405_10...,29.998007,301,1920,1080,10.034000
6,artificial_jewellery,video_20260404_121903.mp4,../data/artificial_jewellery/video_20260404_12...,30.001617,13706,1920,1080,456.842044
7,home_cleaning,video_20260404_073018.mp4,../data/home_cleaning/video_20260404_073018.mp4,30.008047,12196,1920,1080,406.424311
8,home_cleaning,video_20260404_074030.mp4,../data/home_cleaning/video_20260404_074030.mp4,30.007559,7725,1920,1080,257.435133
9,shop,video_20260405_163219_edit.mp4,../data/shop/video_20260405_163219_edit.mp4,30.000136,6639,1920,1080,221.299000


In [6]:
output_csv = "../outputs/video_metadata.csv"
os.makedirs("../outputs", exist_ok=True)

df_videos.to_csv(output_csv, index=False)
print("Saved metadata to:", output_csv)

Saved metadata to: ../outputs/video_metadata.csv
